# Memory-Aware Layout Transformations

Restructuring operator sequences through fusion and simplification, optimizing the physical layout of tensor data in memory is a significant graph-level optimization technique. How tensor dimensions are ordered in linear memory directly impacts data locality, cache utilization, vectorization efficiency, and ultimately, the performance of compute kernels on specific hardware targets. Memory-aware layout transformations analyze the computation graph and the target hardware to choose optimal data layouts, inserting explicit transpose operations only when necessary and beneficial.

## The Significance of Data Layout

##### Consider a typical 4D tensor used in computer vision: Batch ($N$), Channels ($C$), Height ($H$), Width ($W$). Two predominant memory layouts are:

1. $\text{NCHW (Channels First)}$: 
    - Data is stored contiguously along the Width dimension, then Height, then Channels, then Batch. 
    - The memory address for element $tensor[n][c][h][w]$ can be calculated as:
    $$
    offset = n\times C \times H \times W + c \times H \times W + h \times W + w
    $$
    - This layout is common in frameworks like $PyTorch$ and is often favored by $GPU$ libraries like $cuDNN$ for certain operations.

2. $\text{NHWC (Channels Last)}$:
    - Data is stored contiguously along the Channel dimension, then Width, then Height, then Batch. 
    - The memory address for element $tensor[n][h][w][c]$ can be calculated as:
    $$
    offset = n\times H \times W \times C + h \times W \times C + w \times C + c
    $$
    - This layout is the default in $TensorFlow$ and can be advantageous for $CPU$ execution and certain hardware accelerators.

##### The choice between $\text{NCHW}$ and $\text{NHWC}$ (or other potential layouts for higher dimensions) is not arbitrary. It profoundly affects performance:

- $\text{Cache Locality}$: Operations like convolution involve sliding a filter across spatial dimensions (H, W). If data accessed together spatially is also close together in memory, cache hit rates improve. NHWC often provides better spatial locality for typical CPU convolution loop structures, as elements $[h][w][c=0],[h][w][c=1],...,[h][w][c=C-1]$ are contiguous. NCHW provides better channel locality.

- $\text{Memory Access Coalescing (GPUs)}$: GPUs achieve high memory bandwidth by having threads in a warp access contiguous memory locations simultaneously (coalescing). Kernels designed for NCHW might achieve better coalescing when processing multiple channels for a given spatial location concurrently across threads.

- $\text{Vectorization (SIMD)}$: CPU SIMD instructions (like AVX, NEON) operate on vectors of data. Compilers can often vectorize operations more effectively if the data elements to be processed simultaneously reside in contiguous memory. For operations processing multiple channels at once, NHWC's contiguous channel data can be beneficial for vectorization.

- $\text{Hardware Specialization}$: Some hardware, like NVIDIA's Tensor Cores or Google's TPUs, may have internal structures or instruction sets that operate most efficiently on data presented in a specific layout, often resembling NHWC or tiled variants.

## Performing Layout Transformation

ML compilers implement layout transformation as a graph pass. The process typically involves:

1. $\text{Hardware Profiling/Modeling}$: The compiler needs information about the target hardware's preferred layout for different operations (e.g., Convolution, Pooling, Matrix Multiplication). This might come from built-in heuristics, performance models, or even empirical profiling data.

2. $\text{Operator Annotation}$: Identify operators in the graph that are sensitive to layout (e.g., Convolution) versus those that are layout-agnostic (e.g., element-wise ops like ReLU). Layout-agnostic ops typically propagate the layout preference of their inputs or outputs.

3. $\text{Layout Preference Propagation}$: Starting from layout-sensitive operators and hardware preferences, the compiler attempts to propagate the desired layout through the graph. For instance, if a GPU convolution prefers NCHW inputs and outputs, this preference is pushed to adjacent operators.

4. $\text{Conflict Resolution and Transpose Insertion}$: Conflicts arise when adjacent operators prefer different layouts, or when an operator receives inputs with conflicting layout requirements. The compiler must decide where to insert explicit transpose nodes (e.g., NCHW_to_NHWC or NHWC_to_NCHW). The goal is to minimize the number of transposes, as they incur computational cost and memory traffic. Cost models are used to weigh the benefit of executing an operator in its preferred layout against the cost of the required transposes.

5. $\text{Interaction with Fusion}$: Layout transformation often precedes or is interleaved with operator fusion. Fusing operators that share the same preferred layout can eliminate intermediate tensors and potential transpose operations between them. Conversely, choosing a specific layout might enable or disable certain fusion opportunities.

## Example: Layout choice for Conv $\rightarrow$ ReLU $\rightarrow$ Conv

Consider a simple sequence: Conv1 $\rightarrow$ ReLU $\rightarrow$ Conv2

- Assume the target is a GPU where cuDNN prefers NCHW for convolutions.

- ReLU is layout-agnostic.

The compiler might analyze this as follows:

- Conv1 prefers NCHW input and output.

- ReLU can operate efficiently on NCHW data, propagating the layout.

- Conv2 prefers NCHW input.

In this scenario, if the graph input is already NCHW (or can be transposed cheaply at the start), the compiler would likely maintain the NCHW layout throughout this sequence, avoiding any internal transposes.

![](image1.png)

Example graph transformation maintaining NCHW layout.

Now, suppose Conv2 (perhaps due to its specific dimensions or a different kernel implementation) performs significantly better with NHWC inputs on this hardware.

The compiler's cost model would evaluate:

1. Cost of Conv1(NCHW) $\rightarrow$ ReLU(NCHW) $\rightarrow$ Transpose(NCHW -> NHWC) $\rightarrow$ Conv2(NHWC).

2. Cost of Transpose(Input -> NHWC) $\rightarrow$ Conv1(NHWC) $\rightarrow$ ReLU(NHWC) $\rightarrow$ Conv2(NHWC) (assuming Conv1 can run in NHWC, potentially slower).

3. Other combinations.

If the performance gain of Conv2 in NHWC outweighs the transpose cost, the compiler inserts the transpose.

![](image2.png)

Example graph transformation inserting a transpose for optimal Conv2 execution.

## Challenges

Choosing optimal layouts is complex:

- $\text{Transpose Costs}$: Transposes are not free. Minimizing their insertion is important.

- $\text{Hardware Diversity}$: Even on the same hardware, different operators (or the same operator with different parameters) might prefer different layouts.

- $\text{Global vs Local Optimization}$: Finding the globally optimal layout assignment for an entire complex graph is computationally hard; compilers often rely on heuristics.

Memory-aware layout transformation is a powerful graph optimization technique. By intelligently selecting data layouts based on hardware characteristics and operator sequences, compilers can significantly improve cache performance, memory bandwidth utilization, and overall execution speed, often working synergistically with operator fusion to minimize overhead and maximize kernel efficiency. Frameworks like MLIR provide infrastructure (e.g., dialect attributes, interfaces) to represent layout information and implement these transformations systematically.